In [ ]:
# =============================
# 🔹 Enhanced Autoencoder Model with Learning Curves & Parameter Analysis
# =============================
import os
import math
import h5py
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
import random
import warnings
import pywt
import time
from scipy import stats

warnings.filterwarnings('ignore')

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Create results directory if it doesn't exist
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)
print(f"Results will be saved in: {os.path.abspath(results_dir)}")

print("Loading Enhanced Autoencoder with Progressive Training...")

# =============================
# 🔹 WAVELET COMPRESSION METHODS
# =============================
WAVELET_TYPES = ['db4', 'sym8', 'coif3']

class WaveletCompressor:
    def __init__(self, wavelet_type, cr):
        self.wavelet_type = wavelet_type
        self.cr = cr
        
    def compress_decompress(self, signal):
        """Compress and decompress signal using wavelet thresholding"""
        try:
            # Perform wavelet decomposition
            coeffs = pywt.wavedec(signal, self.wavelet_type, level=6)
            
            # Calculate threshold based on compression ratio
            all_coeffs = np.concatenate([c.flatten() for c in coeffs])
            sorted_coeffs = np.sort(np.abs(all_coeffs))[::-1]
            
            # Keep only top coefficients to achieve target CR
            target_coeffs = len(signal) // self.cr
            if target_coeffs < 1:
                target_coeffs = 1
                
            if len(sorted_coeffs) > target_coeffs:
                threshold = sorted_coeffs[target_coeffs]
            else:
                threshold = 0
                
            # Apply threshold
            compressed_coeffs = []
            for coeff in coeffs:
                compressed_coeff = coeff * (np.abs(coeff) > threshold)
                compressed_coeffs.append(compressed_coeff)
            
            # Reconstruct signal
            reconstructed = pywt.waverec(compressed_coeffs, self.wavelet_type)
            
            # Ensure same length as original
            if len(reconstructed) > len(signal):
                reconstructed = reconstructed[:len(signal)]
            elif len(reconstructed) < len(signal):
                reconstructed = np.pad(reconstructed, (0, len(signal) - len(reconstructed)))
                
            return reconstructed
            
        except Exception as e:
            print(f"Wavelet compression error: {e}")
            return signal  # Return original if compression fails

# =============================
# 🔹 Load and Filter Metadata
# =============================
csv_file = r"C:\\Users\\hafiz\\Desktop\\Mostafa seismic signals\\seismic data\\merged.csv"
df = pd.read_csv(csv_file, low_memory=False)
df = df[(df.trace_category == 'earthquake_local') &
        (df.source_distance_km <= 60) &
        (df.source_magnitude > 3)]
trace_names = df['trace_name'].to_list()

print(f"Found {len(trace_names)}")

# =============================
# 🔹 Efficient Data Loading - FULL DATASET
# =============================
def load_filtered_z_waveforms(hdf5_file, trace_names, group='data', min_len=1500, max_samples=None):
    """Load waveforms efficiently without normalization - Loads full dataset"""
    data = []
    if max_samples is None:
        max_samples = len(trace_names)
    
    print(f"Loading up to {max_samples} waveforms...")
    
    with h5py.File(hdf5_file, 'r') as f:
        for i, name in enumerate(trace_names):
            if i >= max_samples:
                break
            try:
                waveform = f[group][name][:]
                if waveform.shape[0] >= min_len:
                    data.append(waveform[:min_len, 0])
            except KeyError:
                continue
                
            if (i + 1) % 1000 == 0:
                print(f"Processed {i + 1}/{min(max_samples, len(trace_names))} waveforms")
    
    print(f"Successfully loaded {len(data)} waveforms")
    return np.array(data)

file_name = r"C:\\Users\\hafiz\\Desktop\\Mostafa seismic signals\\seismic data\\merged.hdf5"
waveforms = load_filtered_z_waveforms(file_name, trace_names, max_samples=None)
print(f"Raw waveforms shape: {waveforms.shape}")

# =============================
# 🔹 Memory-Efficient Dataset
# =============================
class EfficientSeismicDataset(Dataset):
    def __init__(self, data):
        self.data = data
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        waveform = self.data[idx]
        z_min = np.min(waveform)
        z_max = np.max(waveform)
        
        if z_max - z_min > 0:
            waveform = (waveform - z_min) / (z_max - z_min)
        else:
            waveform = np.zeros_like(waveform)
        
        return torch.tensor(waveform, dtype=torch.float32).unsqueeze(0)

# =============================
# 🔹 COMPARISON MODELS
# =============================

# -------------------------
# 🔹1- GeneralizedAutoencoder Model
# -------------------------
class GeneralizedAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        latent_dim = 1500 // cr
        
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=9, stride=2, padding=4), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(32 * 375, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32 * 375),
            nn.Unflatten(1, (32, 375)),
            nn.ConvTranspose1d(32, 16, kernel_size=9, stride=2, padding=4, output_padding=1), nn.ReLU(),
            nn.ConvTranspose1d(16, 1, kernel_size=9, stride=2, padding=4, output_padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

# -------------------------
# 🔹2- AE_PureConcat Model
# -------------------------
# Architecture configuration per compression ratio
model_2_architecture = {
    100: {"layers": 2, "feature_maps": [16, 16]},
     60: {"layers": 2, "feature_maps": [16, 16]},
     50: {"layers": 2, "feature_maps": [16, 16]},
     30: {"layers": 2, "feature_maps": [8,  8 ]},
     20: {"layers": 2, "feature_maps": [16, 16]},
     15: {"layers": 2, "feature_maps": [16, 16]},
     10: {"layers": 2, "feature_maps": [16, 16]},
      5: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      3: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
      2: {"layers": 4, "feature_maps": [16, 16, 16, 16]},
}

class AE_PureConcat(nn.Module):
    def __init__(
        self,
        cr: int,
        input_len: int = 1500,
        kernel_size: int = 3,
        activation: nn.Module = nn.ELU,
        dropout: float = 0.1,
    ):
        super().__init__()
        cfg       = model_2_architecture[cr]
        layers    = cfg["layers"]
        fmaps     = cfg["feature_maps"]
        total_lat = input_len // cr

        # Distribute latent dimensions across branches
        base_dim  = total_lat // layers
        rem       = total_lat % layers
        latent_sizes = [base_dim + (1 if i < rem else 0) for i in range(layers)]

        # Encoder branches
        self.branches = nn.ModuleList()
        for i in range(layers):
            branch = nn.Sequential(
                nn.Conv1d(1, fmaps[i], kernel_size, padding=1),
                activation(inplace=True),
                nn.Dropout(dropout),
                nn.Flatten(),
                nn.Linear(input_len * fmaps[i], latent_sizes[i]),
                activation(inplace=True),
            )
            self.branches.append(branch)

        # Decoder: single linear layer to reconstruct full signal
        self.decoder = nn.Linear(total_lat, input_len)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: (batch, 1, input_len)
        zs = [branch(x) for branch in self.branches]     # list of (batch, latent_i)
        z  = torch.cat(zs, dim=1)                        # (batch, total_lat)
        out = self.decoder(z)                            # (batch, input_len)
        return out.unsqueeze(1)                          # → (batch, 1, input_len)

# -------------------------
# 🔹3- HEAVY CAPACITY ARCHITECTURE (EXACT ABLATION STUDY PARAMETERS)
# -------------------------
class AttentionBlock(nn.Module):
    """Simplified channel attention mechanism - EXACT ABLATION VERSION"""
    def __init__(self, channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Conv1d(channels, max(4, channels//8), kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv1d(max(4, channels//8), channels, kernel_size=1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        attention_weights = self.attention(x)
        return x * attention_weights

class EnhancedResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, use_attention=False):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.elu = nn.ELU(inplace=True)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
        
        self.attention = AttentionBlock(out_channels) if use_attention else nn.Identity()
            
    def forward(self, x):
        residual = self.shortcut(x)
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.elu(out)
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.elu(out)
        out = self.attention(out)
        return out

class HeavyAdvancedResidualAutoencoder(nn.Module):
    def __init__(self, cr):
        super().__init__()
        self.cr = cr
        self.target_length = max(16, 1500 // cr)
        
        # EXACT ABLATION STUDY ARCHITECTURE
        # HEAVY CAPACITY ENCODER (2x original channels)
        encoder_layers = [
            # Initial convolution: 1 → 64 channels (2x original) - EXACT ABLATION PARAMETERS
            nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7),
            nn.BatchNorm1d(64),
            nn.ELU(inplace=True),
            AttentionBlock(64),
            
            # Residual Block 1: 64 → 128 channels - EXACT ABLATION PARAMETERS
            EnhancedResidualBlock(64, 128, stride=2, use_attention=True),
            
            # Residual Block 2: 128 → 256 channels - EXACT ABLATION PARAMETERS
            EnhancedResidualBlock(128, 256, stride=2, use_attention=True),
        ]
        
        # Extra layer for high compression ratios (CR > 30) - EXACT ABLATION LOGIC
        if cr > 30:
            encoder_layers.append(EnhancedResidualBlock(256, 512, stride=2, use_attention=True))
            final_channels = 512
        else:
            final_channels = 256
        
        # Final compression
        encoder_layers.append(nn.AdaptiveAvgPool1d(self.target_length))
        self.encoder = nn.Sequential(*encoder_layers)
        
        # Calculate upsampling factors - EXACT ABLATION METHOD
        self.upsample_factors = self.calculate_upsample_factors()
        
        # HEAVY CAPACITY DECODER - EXACT ABLATION PARAMETERS
        decoder_layers = []
        current_channels = final_channels
        
        for i, factor in enumerate(self.upsample_factors):
            next_channels = max(32, current_channels // 2)  # Increased minimum channels - EXACT ABLATION VALUE
            
            decoder_layers.extend([
                nn.Upsample(scale_factor=factor, mode='linear', align_corners=False),
                EnhancedResidualBlock(current_channels, next_channels, stride=1, use_attention=True)
            ])
            current_channels = next_channels
        
        # Final layers - EXACT ABLATION PARAMETERS
        decoder_layers.extend([
            nn.Conv1d(current_channels, 1, kernel_size=15, padding=7),
            nn.Sigmoid()
        ])
        
        self.decoder = nn.Sequential(*decoder_layers)
        
        # Test architecture
        self.test_dimensions()
    
    def calculate_upsample_factors(self):
        """Calculate exact upsampling factors safely - EXACT ABLATION METHOD"""
        current_length = self.target_length
        factors = []
        
        while current_length < 1500:
            remaining_ratio = 1500 / current_length
            factor = min(2.0, remaining_ratio)
            factors.append(factor)
            current_length = int(current_length * factor)
            
            if len(factors) >= 5:
                break
        
        if current_length != 1500:
            final_factor = 1500 / current_length
            factors.append(final_factor)
        
        return factors
    
    def test_dimensions(self):
        """Test if the architecture can handle the input/output dimensions"""
        try:
            with torch.no_grad():
                test_input = torch.randn(1, 1, 1500)
                encoded = self.encoder(test_input)
                decoded = self.decoder(encoded)
                
                if decoded.shape[2] != 1500:
                    final_upsample = nn.Upsample(size=1500, mode='linear', align_corners=False)
                    old_decoder = self.decoder
                    self.decoder = nn.Sequential(old_decoder, final_upsample)
                    decoded = self.decoder(encoded)
                
                assert decoded.shape[2] == 1500, f"Output shape {decoded.shape[2]} != 1500"
                print(f"✓ Heavy Architecture test passed: Input 1500 -> Compressed {encoded.shape[2]} -> Output {decoded.shape[2]}")
                
        except Exception as e:
            print(f"✗ Heavy Architecture test failed: {e}")
            self.create_fallback_architecture()
    
    def create_fallback_architecture(self):
        """Create a simple fallback architecture"""
        print("Creating fallback architecture for Heavy model...")
        
        self.encoder = nn.Sequential(
            nn.Conv1d(1, 64, kernel_size=15, stride=2, padding=7),
            nn.ELU(),
            nn.Conv1d(64, 128, kernel_size=9, stride=2, padding=4),
            nn.ELU(),
            nn.AdaptiveAvgPool1d(self.target_length)
        )
        
        self.decoder = nn.Sequential(
            nn.Upsample(size=1500, mode='linear', align_corners=False),
            nn.Conv1d(128, 64, kernel_size=9, padding=4),
            nn.ELU(),
            nn.Conv1d(64, 1, kernel_size=15, padding=5),
            nn.Sigmoid()
        )
        
        print("✓ Heavy Fallback architecture created successfully")
    
    def forward(self, x):
        encoded = self.encoder(x)
        return self.decoder(encoded)

# -------------------------
# 🔹4- Wavelet Compression Models
# -------------------------
class WaveletDB4:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'db4'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('db4', self.cr).compress_decompress(signal)

class WaveletSYM8:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'sym8'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('sym8', self.cr).compress_decompress(signal)

class WaveletCOIF3:
    def __init__(self, cr):
        self.cr = cr
        self.wavelet_type = 'coif3'
        
    def compress_decompress(self, signal):
        return WaveletCompressor('coif3', self.cr).compress_decompress(signal)

# =============================
# 🔹  TRAINING CONFIGURATION
# =============================
def train_model_heavy_ablation(model, train_loader, val_loader, model_name, cr, max_epochs=None):
    """EXACT ABLATION STUDY TRAINING PARAMETERS FOR HEAVY MODEL"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} CR={cr} on {device} - USING EXACT ABLATION PARAMETERS")
    
    # EXACT ABLATION STUDY EPOCH ASSIGNMENT
    if max_epochs is None:
        if cr <= 5:
            max_epochs = 120    
        elif cr <= 15:
            max_epochs = 100    
        elif cr <= 30:
            max_epochs = 100    
        else:
            max_epochs = 100    
    
    print(f"Using {max_epochs} epochs for CR={cr} (EXACT Ablation Study Parameters)")
    
    model = model.to(device)
    
    # EXACT ABLATION STUDY LEARNING RATES - CORRECTED TO MATCH ORIGINAL ABLATION
    if cr <= 3:
        lr = 0.0002    
    elif cr <= 5:
        lr = 0.0002    
    elif cr <= 15:
        lr = 0.0002    
    else:
        lr = 0.0002    
    
    print(f"Using learning rate: {lr} (EXACT Ablation Study Value)")
    
    # EXACT ABLATION STUDY OPTIMIZER CONFIGURATION
    optimizer = torch.optim.Adam(
        model.parameters(), 
        lr=lr, 
        weight_decay=1e-5
    )
    
    # EXACT ABLATION STUDY SCHEDULER
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, 
        mode='min',
        factor=0.5,
        patience=5,
        verbose=True,
        min_lr=1e-6
    )
    
    # EXACT ABLATION STUDY EARLY STOPPING CONFIG
    best_val_loss = float('inf')
    patience = 8
    patience_counter = 0
    best_model_state = None
    
    train_losses = []
    val_losses = []
    
    print(f"Starting Heavy Model training with EXACT ABLATION PARAMETERS: {max_epochs} epochs, LR={lr}, patience={patience}")
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            inputs = batch.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            
            loss = nn.MSELoss()(inputs, outputs)
                
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
            
            if batch_idx % 500 == 0:
                print(f"Epoch {epoch+1}, Batch {batch_idx}, Loss: {loss.item():.6f}")
        
        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device)
                outputs = model(inputs)
                loss = nn.MSELoss()(inputs, outputs)
                val_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{max_epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}, LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # EARLY STOPPING CHECK
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f'best_{model_name}_cr{cr}.pth')
            print(f"✓ New best model saved (val_loss: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
        
        if epoch % 2 == 0 and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"✓ Loaded best model with val_loss: {best_val_loss:.6f}")
    elif os.path.exists(f'best_{model_name}_cr{cr}.pth'):
        model.load_state_dict(torch.load(f'best_{model_name}_cr{cr}.pth', map_location=device))
        print("✓ Loaded saved model from file")
    
    return model, train_losses, val_losses

def train_model_general(model, train_loader, val_loader, model_name, cr, max_epochs=None):
    """General training function for other models"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training {model_name} CR={cr} on {device}")
    
    if max_epochs is None:
        max_epochs = 100
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_loss = float('inf')
    patience = 8
    patience_counter = 0
    best_model_state = None
    
    train_losses = []
    val_losses = []
    
    for epoch in range(max_epochs):
        model.train()
        train_loss = 0
        
        for batch_idx, batch in enumerate(train_loader):
            inputs = batch.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = nn.MSELoss()(inputs, outputs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for batch in val_loader:
                inputs = batch.to(device)
                outputs = model(inputs)
                loss = nn.MSELoss()(inputs, outputs)
                val_loss += loss.item()
        
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        train_losses.append(avg_train_loss)
        val_losses.append(avg_val_loss)
        
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{max_epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
            torch.save(model.state_dict(), f'best_{model_name}_cr{cr}.pth')
            print(f"✓ New best model saved (val_loss: {best_val_loss:.6f})")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break
    
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, train_losses, val_losses

# =============================
# 🔹 Enhanced Visualization with Learning Curves & Parameter Analysis
# =============================
def plot_waveforms_comprehensive(original, reconstructed, cr, model_name, save_figures=True):
    """
    Enhanced visualization with time and frequency domains for ALL neural network models
    """
    num_samples = min(6, len(original))
    
    # Create figure with subplots
    fig, axes = plt.subplots(num_samples, 2, figsize=(15, 3*num_samples))
    
    # Handle single sample case
    if num_samples == 1:
        axes = np.array([axes])
    
    for i in range(num_samples):
        # Ensure we have 1D arrays for processing
        orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
        recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]
        
        # Ensure both signals have the same length (1500)
        min_len = min(len(orig_signal), len(recon_signal))
        orig_signal = orig_signal[:min_len]
        recon_signal = recon_signal[:min_len]
        
        # Time domain plot
        axes[i, 0].plot(orig_signal, 'b-', label='Original', linewidth=1.5, alpha=0.8)
        axes[i, 0].plot(recon_signal, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
        axes[i, 0].set_title(f'{model_name} - CR={cr} - Sample {i+1}', fontsize=12, fontweight='bold')
        axes[i, 0].set_xlabel('Time Samples')
        axes[i, 0].set_ylabel('Amplitude (Normalized)')
        axes[i, 0].legend()
        axes[i, 0].grid(True, alpha=0.3)
        
        # Frequency domain plot - FIXED DIMENSIONALITY ISSUE
        try:
            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))
            
            # Only plot positive frequencies
            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]
            
            axes[i, 1].semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=1.5, alpha=0.8)
            axes[i, 1].semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=1.5, alpha=0.8)
            axes[i, 1].set_title('Frequency Spectrum', fontsize=12, fontweight='bold')
            axes[i, 1].set_xlabel('Frequency (Normalized)')
            axes[i, 1].set_ylabel('Magnitude (log scale)')
            axes[i, 1].legend()
            axes[i, 1].grid(True, alpha=0.3)
        except Exception as e:
            print(f"Warning: Frequency plot failed for sample {i+1}: {e}")
            axes[i, 1].text(0.5, 0.5, f'Frequency plot failed\n{str(e)}', 
                           ha='center', va='center', transform=axes[i, 1].transAxes)
            axes[i, 1].set_title('Frequency Spectrum - Failed', fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    
    if save_figures:
        # Create model-specific directory
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        
        # Save figures in multiple formats
        filename_base = f"{model_name}_CR{cr}_comprehensive"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
        print(f"✓ Saved comprehensive waveforms to: {png_path} and {eps_path}")
    
    plt.show()
    plt.close()

def plot_frequency_spectrum_all_models(original, reconstructed, cr, model_name, save_figures=True):
    """
    Save frequency spectrum plots for ALL models (including autoencoders) for first six samples
    """
    num_samples = min(6, len(original))
    
    for i in range(num_samples):
        try:
            # Ensure we have 1D arrays for processing
            orig_signal = original[i].flatten() if len(original[i].shape) > 1 else original[i]
            recon_signal = reconstructed[i].flatten() if len(reconstructed[i].shape) > 1 else reconstructed[i]
            
            # Ensure both signals have the same length
            min_len = min(len(orig_signal), len(recon_signal))
            orig_signal = orig_signal[:min_len]
            recon_signal = recon_signal[:min_len]
            
            # Create figure for frequency spectrum only
            plt.figure(figsize=(10, 6))
            
            # Calculate frequency spectra
            orig_fft = np.abs(np.fft.fft(orig_signal))
            recon_fft = np.abs(np.fft.fft(recon_signal))
            freq = np.fft.fftfreq(len(orig_signal))
            
            # Only plot positive frequencies
            positive_freq_mask = freq >= 0
            freq_positive = freq[positive_freq_mask]
            orig_fft_positive = orig_fft[positive_freq_mask]
            recon_fft_positive = recon_fft[positive_freq_mask]
            
            # Plot frequency spectrum
            plt.semilogy(freq_positive, orig_fft_positive, 'b-', label='Original', linewidth=2, alpha=0.8)
            plt.semilogy(freq_positive, recon_fft_positive, 'r-', label='Reconstructed', linewidth=2, alpha=0.8)
            plt.title(f'{model_name} - CR={cr} - Sample {i+1} - Frequency Spectrum', fontsize=14, fontweight='bold')
            plt.xlabel('Frequency (Normalized)', fontsize=12)
            plt.ylabel('Magnitude (log scale)', fontsize=12)
            plt.legend(fontsize=11)
            plt.grid(True, alpha=0.3)
            
            if save_figures:
                # Create frequency spectrum directory
                freq_dir = os.path.join(results_dir, model_name, "frequency_spectrum")
                os.makedirs(freq_dir, exist_ok=True)
                
                # Save in both PNG and EPS formats
                filename_base = f"{model_name}_CR{cr}_sample{i+1}_freq_spectrum"
                png_path = os.path.join(freq_dir, f"{filename_base}.png")
                eps_path = os.path.join(freq_dir, f"{filename_base}.eps")
                
                plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
                plt.savefig(eps_path, format='eps', bbox_inches='tight', facecolor='white')
                print(f"✓ Saved frequency spectrum for sample {i+1} to: {png_path}")
            
            plt.close()
            
        except Exception as e:
            print(f"Error creating frequency spectrum for {model_name} CR={cr} sample {i+1}: {e}")
            plt.close()

def plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True):
    """Plot training and validation loss history"""
    if not train_losses or len(train_losses) == 0:
        print(f"No training history available for {model_name} CR={cr}")
        return
        
    plt.figure(figsize=(12, 8))
    epochs = range(1, len(train_losses) + 1)
    
    plt.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2, alpha=0.8)
    plt.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2, alpha=0.8)
    
    plt.title(f'{model_name} - CR={cr} - Training History', fontsize=16, fontweight='bold')
    plt.xlabel('Epoch', fontsize=14)
    plt.ylabel('MSE Loss', fontsize=14)
    plt.yscale('log')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    
    # Add final loss values to the plot
    final_train_loss = train_losses[-1]
    final_val_loss = val_losses[-1] if val_losses else train_losses[-1]
    plt.annotate(f'Final Train: {final_train_loss:.6f}\nFinal Val: {final_val_loss:.6f}', 
                xy=(0.02, 0.98), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                fontsize=10, ha='left', va='top')
    
    if save_figures:
        model_dir = os.path.join(results_dir, model_name)
        os.makedirs(model_dir, exist_ok=True)
        
        filename_base = f"{model_name}_CR{cr}_training_history"
        png_path = os.path.join(model_dir, f"{filename_base}.png")
        eps_path = os.path.join(model_dir, f"{filename_base}.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved training history to: {png_path}")
    
    plt.show()
    plt.close()

def plot_parameter_comparison(results_df, save_figures=True):
    """Plot parameter count comparison across all models and compression ratios"""
    neural_models = [model for model in results_df['Model'].unique() if 'Wavelet' not in model]
    cr_values = sorted(results_df['CR'].unique())
    
    plt.figure(figsize=(14, 8))
    
    # Create color map for models
    colors = plt.cm.Set3(np.linspace(0, 1, len(neural_models)))
    
    for i, model in enumerate(neural_models):
        param_counts = []
        for cr in cr_values:
            model_cr_data = results_df[(results_df['Model'] == model) & (results_df['CR'] == cr)]
            if not model_cr_data.empty:
                param_counts.append(model_cr_data['Parameter_Count'].values[0] / 1e6)
            else:
                param_counts.append(np.nan)
        
        plt.semilogx(cr_values, param_counts, 'o-', color=colors[i], label=model, 
                    markersize=8, linewidth=2, markerfacecolor=colors[i], markeredgecolor='black')
    
    plt.title('Parameter Count Comparison Across Models', fontsize=16, fontweight='bold')
    plt.xlabel('Compression Ratio', fontsize=14)
    plt.ylabel('Parameter Count (Millions)', fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12, loc='upper right')
    
    # Add text annotation with model statistics
    stats_text = "Model Parameter Statistics:\n"
    for model in neural_models:
        model_data = results_df[results_df['Model'] == model]
        avg_params = model_data['Parameter_Count'].mean() / 1e6
        max_params = model_data['Parameter_Count'].max() / 1e6
        min_params = model_data['Parameter_Count'].min() / 1e6
        stats_text += f"{model}: {avg_params:.2f}M avg\n"
    
    plt.annotate(stats_text, xy=(0.02, 0.02), xycoords='axes fraction',
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8),
                fontsize=10, ha='left', va='bottom')
    
    if save_figures:
        analysis_dir = os.path.join(results_dir, "model_analysis")
        os.makedirs(analysis_dir, exist_ok=True)
        
        png_path = os.path.join(analysis_dir, "parameter_count_comparison.png")
        eps_path = os.path.join(analysis_dir, "parameter_count_comparison.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved parameter count comparison to: {png_path}")
    
    plt.show()
    plt.close()

def plot_parameter_vs_performance(results_df, save_figures=True):
    """Plot parameter count vs performance metrics"""
    neural_models = [model for model in results_df['Model'].unique() if 'Wavelet' not in model]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    
    metrics = ['SNR', 'PSNR', 'SSIM', 'MSE']
    
    for i, metric in enumerate(metrics):
        for model in neural_models:
            model_data = results_df[results_df['Model'] == model]
            if not model_data.empty:
                params = model_data['Parameter_Count'] / 1e6
                performance = model_data[metric]
                
                markers = ['o', 's', 'D', '^', 'v', '<', '>']
                marker = markers[neural_models.index(model) % len(markers)]
                
                axes[i].scatter(params, performance, label=model, marker=marker, s=80, alpha=0.7)
        
        axes[i].set_title(f'{metric} vs Parameter Count', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Parameter Count (Millions)', fontsize=12)
        axes[i].set_ylabel(metric, fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(fontsize=10)
    
    plt.tight_layout()
    
    if save_figures:
        analysis_dir = os.path.join(results_dir, "model_analysis")
        os.makedirs(analysis_dir, exist_ok=True)
        
        png_path = os.path.join(analysis_dir, "parameter_vs_performance.png")
        eps_path = os.path.join(analysis_dir, "parameter_vs_performance.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')  # FIXED: dpi not dio
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved parameter vs performance analysis to: {png_path}")
    
    plt.show()
    plt.close()

def plot_comprehensive_training_analysis(results_df, training_histories, save_figures=True):
    """Plot comprehensive training analysis across all models"""
    neural_models = [model for model in results_df['Model'].unique() if 'Wavelet' not in model]
    
    if not training_histories:
        print("No training histories available for comprehensive analysis")
        return
    
    # Plot final training losses comparison
    plt.figure(figsize=(12, 8))
    
    final_train_losses = []
    final_val_losses = []
    model_names = []
    
    for model in neural_models:
        model_histories = [hist for key, hist in training_histories.items() if model in key]
        if model_histories:
            train_losses, val_losses = model_histories[-1]
            if train_losses:
                final_train_losses.append(train_losses[-1])
                final_val_losses.append(val_losses[-1] if val_losses else train_losses[-1])
                model_names.append(model)
    
    if final_train_losses:
        x_pos = np.arange(len(model_names))
        width = 0.35
        
        plt.bar(x_pos - width/2, final_train_losses, width, label='Final Train Loss', alpha=0.8)
        plt.bar(x_pos + width/2, final_val_losses, width, label='Final Val Loss', alpha=0.8)
        
        plt.xlabel('Models', fontsize=14)
        plt.ylabel('Final MSE Loss', fontsize=14)
        plt.title('Final Training vs Validation Loss Comparison', fontsize=16, fontweight='bold')
        plt.xticks(x_pos, model_names, rotation=45)
        plt.legend()
        plt.grid(True, alpha=0.3, axis='y')
        plt.yscale('log')
        
        if save_figures:
            analysis_dir = os.path.join(results_dir, "model_analysis")
            os.makedirs(analysis_dir, exist_ok=True)
            
            png_path = os.path.join(analysis_dir, "final_loss_comparison.png")
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            print(f"✓ Saved final loss comparison to: {png_path}")
        
        plt.show()
        plt.close()

# =============================
# 🔹 Evaluation Metrics (PSNR and SSIM)
# =============================
def calculate_psnr(original, reconstructed):
    """Calculate PSNR between original and reconstructed signals"""
    mse = np.mean((original - reconstructed) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 1.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

def calculate_ssim_1d(original, reconstructed, data_range=1.0):
    """Calculate SSIM for 1D signals"""
    if len(original.shape) > 1:
        original = original.flatten()
    if len(reconstructed.shape) > 1:
        reconstructed = reconstructed.flatten()
    
    return ssim(original, reconstructed, data_range=data_range)

# =============================
# 🔹 Model Size Calculation
# =============================
def calculate_model_size(model):
    """Calculate model size in MB"""
    param_size = 0
    for param in model.parameters():
        param_size += param.nelement() * param.element_size()
    buffer_size = 0
    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()
    
    size_mb = (param_size + buffer_size) / 1024**2
    return size_mb

# =============================
# 🔹 Model Evaluation Function
# =============================
def evaluate_model(model, test_data, cr, model_name):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    if "Wavelet" in model_name:
        # Handle wavelet compression evaluation
        x = test_data[:10]
        recon = []
        for signal in x:
            signal_1d = signal[0] if len(signal.shape) > 1 else signal
            reconstructed = model.compress_decompress(signal_1d)
            recon.append(reconstructed)
        recon = np.array(recon)
        x_processed = x[:, 0, :] if len(x.shape) > 2 else x
    else:
        # Handle neural network models
        model.eval()
        with torch.no_grad():
            test_samples = test_data[:10]
            inputs = torch.tensor(test_samples, dtype=torch.float32).to(device)
            recon = model(inputs).cpu().numpy()
            x_processed = inputs.cpu().numpy()[:, 0, :]

    # Calculate metrics
    orig_flat = x_processed.flatten()
    recon_flat = recon.flatten()
    
    mse = np.mean((orig_flat - recon_flat) ** 2)
    signal_power = np.mean(orig_flat ** 2)
    snr = 10 * math.log10(signal_power / mse) if mse > 0 else float('inf')
    
    if np.std(orig_flat) > 0 and np.std(recon_flat) > 0:
        correlation = np.corrcoef(orig_flat, recon_flat)[0, 1]
    else:
        correlation = 0
    
    mae = np.mean(np.abs(orig_flat - recon_flat))
    psnr = calculate_psnr(orig_flat, recon_flat)
    ssim_value = calculate_ssim_1d(orig_flat, recon_flat)
    
    # Calculate compression ratio
    if "Wavelet" in model_name:
        actual_cr = cr
    else:
        test_input = torch.randn(1, 1, 1500).to(device)
        compressed = model.encoder(test_input) if hasattr(model, 'encoder') else None
        
        if compressed is not None:
            actual_cr = 1500 / compressed.shape[-1]
        else:
            actual_cr = cr
    
    cr_error = abs(actual_cr - cr) / cr * 100
    
    # Calculate model complexity for neural networks
    if "Wavelet" not in model_name:
        model_size = calculate_model_size(model)
        total_params = sum(p.numel() for p in model.parameters())
    else:
        model_size = 0
        total_params = 0
    
    return {
        "MSE": mse,
        "SNR": snr,
        "Correlation": correlation,
        "MAE": mae,
        "PSNR": psnr,
        "SSIM": ssim_value,
        "CR_Error": cr_error,
        "Model_Size_MB": model_size,
        "Parameter_Count": total_params,
        "Model": model_name,
        "CR": cr,
        "Actual_CR": actual_cr,
        "Reconstructions": (x_processed, recon)
    }

# =============================
# 🔹 Statistical Significance Testing
# =============================
def statistical_significance_test(results_df, metric='SNR'):
    models = results_df['Model'].unique()
    p_values = {}
    
    for i, model1 in enumerate(models):
        for model2 in models[i+1:]:
            data1 = results_df[results_df['Model'] == model1][metric]
            data2 = results_df[results_df['Model'] == model2][metric]
            
            t_stat, p_value = stats.ttest_ind(data1, data2)
            p_values[f"{model1}_vs_{model2}"] = p_value
    
    return p_values

# =============================
# 🔹 Enhanced Plotting Functions
# =============================
def plot_metrics_comparison(results_df, save_figures=True):
    models = results_df['Model'].unique()
    metrics = ['SNR', 'Correlation', 'PSNR', 'SSIM', 'MSE', 'MAE']
    
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    axes = axes.flatten()
    
    for i, metric in enumerate(metrics):
        for model in models:
            model_data = results_df[results_df['Model'] == model]
            if metric in ['MSE', 'MAE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'o-', label=model, markersize=8, linewidth=2)
        
        axes[i].set_title(f'{metric} vs Compression Ratio', fontsize=14, fontweight='bold')
        axes[i].set_xlabel('Compression Ratio', fontsize=12)
        axes[i].set_ylabel(metric, fontsize=12)
        axes[i].grid(True, alpha=0.3)
        axes[i].legend(fontsize=10)
    
    plt.tight_layout()
    
    if save_figures:
        png_path = os.path.join(results_dir, "comprehensive_metrics_comparison.png")
        eps_path = os.path.join(results_dir, "comprehensive_metrics_comparison.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved comprehensive metrics comparison to: {png_path}")
    
    plt.show()
    plt.close()

def plot_model_comparison(results_df, metric, save_figures=True):
    """Plot comparison of a specific metric across all models"""
    models = results_df['Model'].unique()
    cr_values = sorted(results_df['CR'].unique())
    
    plt.figure(figsize=(12, 8))
    
    for model in models:
        model_metrics = []
        for cr in cr_values:
            model_cr_data = results_df[(results_df['Model'] == model) & (results_df['CR'] == cr)]
            if not model_cr_data.empty:
                model_metrics.append(model_cr_data[metric].values[0])
            else:
                model_metrics.append(np.nan)
        
        if metric in ['MSE', 'MAE']:
            plt.loglog(cr_values, model_metrics, 'o-', label=model, markersize=8, linewidth=2)
        else:
            plt.semilogx(cr_values, model_metrics, 'o-', label=model, markersize=8, linewidth=2)
    
    plt.title(f'{metric} Comparison Across Models', fontsize=16, fontweight='bold')
    plt.xlabel('Compression Ratio', fontsize=14)
    plt.ylabel(metric, fontsize=14)
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=12)
    
    if save_figures:
        metric_dir = os.path.join(results_dir, "metric_comparisons")
        os.makedirs(metric_dir, exist_ok=True)
        
        png_path = os.path.join(metric_dir, f"{metric}_comparison.png")
        eps_path = os.path.join(metric_dir, f"{metric}_comparison.eps")
        
        plt.savefig(png_path, dpi=300, bbox_inches='tight')
        plt.savefig(eps_path, format='eps', bbox_inches='tight')
        print(f"✓ Saved {metric} comparison to: {png_path}")
    
    plt.show()
    plt.close()

def plot_individual_model_metrics(results_df, save_figures=True):
    """Plot metrics for each individual model"""
    models = results_df['Model'].unique()
    metrics = ['SNR', 'PSNR', 'SSIM', 'MSE']
    
    for model in models:
        model_data = results_df[results_df['Model'] == model]
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()
        
        for i, metric in enumerate(metrics):
            if metric in ['MSE']:
                axes[i].loglog(model_data['CR'], model_data[metric], 'bo-', markersize=8, linewidth=2)
            else:
                axes[i].semilogx(model_data['CR'], model_data[metric], 'bo-', markersize=8, linewidth=2)
            
            axes[i].set_title(f'{model} - {metric} vs CR', fontsize=14, fontweight='bold')
            axes[i].set_xlabel('Compression Ratio', fontsize=12)
            axes[i].set_ylabel(metric, fontsize=12)
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        
        if save_figures:
            model_metrics_dir = os.path.join(results_dir, model, "metrics")
            os.makedirs(model_metrics_dir, exist_ok=True)
            
            png_path = os.path.join(model_metrics_dir, f"{model}_metrics_summary.png")
            eps_path = os.path.join(model_metrics_dir, f"{model}_metrics_summary.eps")
            
            plt.savefig(png_path, dpi=300, bbox_inches='tight')
            plt.savefig(eps_path, format='eps', bbox_inches='tight')
            print(f"✓ Saved {model} metrics summary to: {png_path}")
        
        plt.show()
        plt.close()

# =============================
# 🔹 Main Execution
# =============================
def main():
    # Create dataset
    dataset = EfficientSeismicDataset(waveforms)
    train_size = int(0.9 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = train_test_split(dataset, train_size=train_size, test_size=val_size, random_state=42)

    # Data loaders
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)

    # Test data
    test_data = np.array([val_dataset[i] for i in range(min(20, len(val_dataset)))])
    test_data_np = test_data[:, 0, :]

    print(f"Training on {len(train_dataset)} samples, testing on {len(test_data)} samples")
    print(f"Full dataset size: {len(dataset)} waveforms")

    # Compression ratios
    compression_ratios = [2, 3, 5, 10, 15, 20, 30, 50, 60, 100]
    results = []
    training_histories = {}

    print("Starting Model Comparison Training...")
    print("="*80)

    # Define models to compare
    models_to_compare = [
        ("HeavyAdvancedResidual", HeavyAdvancedResidualAutoencoder),
        ("GeneralizedAutoencoder", GeneralizedAutoencoder),
        ("AE_PureConcat", AE_PureConcat),
        ("Wavelet_DB4", WaveletDB4),
        ("Wavelet_SYM8", WaveletSYM8),
        ("Wavelet_COIF3", WaveletCOIF3)
    ]

    # Train and evaluate all models
    for model_name, model_class in models_to_compare:
        print(f"\n=== Processing {model_name} Model ===")
        
        for cr in compression_ratios:
            print(f"\nProcessing {model_name} CR={cr}")
            print("-" * 40)
            
            try:
                if "Wavelet" in model_name:
                    model = model_class(cr)
                    metrics = evaluate_model(model, test_data_np, cr, model_name)
                    results.append(metrics)
                    
                    orig, recon = metrics["Reconstructions"]
                    plot_waveforms_comprehensive(orig, recon, cr, model_name, save_figures=True)
                    plot_frequency_spectrum_all_models(orig, recon, cr, model_name, save_figures=True)
                    
                    print(f"{model_name} CR={cr}: SNR={metrics['SNR']:.2f}dB, "
                          f"Corr={metrics['Correlation']:.4f}, MSE={metrics['MSE']:.6f}, "
                          f"PSNR={metrics['PSNR']:.2f}dB, SSIM={metrics['SSIM']:.4f}")
                else:
                    model_file = f'best_{model_name}_cr{cr}.pth'
                    if os.path.exists(model_file):
                        os.remove(model_file)
                    
                    if model_name == "AE_PureConcat" and cr not in model_2_architecture:
                        print(f"Skipping CR={cr} for AE_PureConcat (not in architecture config)")
                        continue
                        
                    model = model_class(cr)
                    
                    if model_name == "HeavyAdvancedResidual":
                        trained_model, train_losses, val_losses = train_model_heavy_ablation(
                            model, train_loader, val_loader, model_name, cr
                        )
                    else:
                        trained_model, train_losses, val_losses = train_model_general(
                            model, train_loader, val_loader, model_name, cr
                        )
                    
                    training_histories[f"{model_name}_CR{cr}"] = (train_losses, val_losses)
                    
                    plot_training_history(train_losses, val_losses, model_name, cr, save_figures=True)
                    
                    metrics = evaluate_model(trained_model, test_data, cr, model_name)
                    results.append(metrics)
                    
                    orig, recon = metrics["Reconstructions"]
                    plot_waveforms_comprehensive(orig, recon, cr, model_name, save_figures=True)
                    plot_frequency_spectrum_all_models(orig, recon, cr, model_name, save_figures=True)
                    
                    print(f"{model_name} CR={cr}: Actual CR={metrics['Actual_CR']:.2f}, SNR={metrics['SNR']:.2f}dB, "
                          f"Corr={metrics['Correlation']:.4f}, MSE={metrics['MSE']:.6f}, "
                          f"PSNR={metrics['PSNR']:.2f}dB, SSIM={metrics['SSIM']:.4f}, "
                          f"Params={metrics['Parameter_Count']:,}, Size={metrics['Model_Size_MB']:.2f}MB")
                
            except Exception as e:
                print(f"Error processing {model_name} CR={cr}: {e}")
                import traceback
                traceback.print_exc()
                continue

    # Save results
    if results:
        results_df = pd.DataFrame(results)
        csv_path = os.path.join(results_dir, 'model_comparison_results.csv')
        results_df.to_csv(csv_path, index=False)
        print(f"\n✓ Results saved to: {csv_path}")
        
        print("\nComprehensive Results Summary:")
        print("=" * 120)
        print(results_df.round(4))
        
        # Generate comprehensive visualizations
        plot_metrics_comparison(results_df, save_figures=True)
        
        for metric in ['SNR', 'PSNR', 'SSIM', 'MSE']:
            plot_model_comparison(results_df, metric, save_figures=True)
        
        plot_individual_model_metrics(results_df, save_figures=True)
        
        # Parameter count analysis
        plot_parameter_comparison(results_df, save_figures=True)
        plot_parameter_vs_performance(results_df, save_figures=True)
        plot_comprehensive_training_analysis(results_df, training_histories, save_figures=True)
        
        # Statistical significance testing
        print("\nStatistical Significance Testing (SNR):")
        p_values = statistical_significance_test(results_df, 'SNR')
        for comparison, p_value in p_values.items():
            significance = "***" if p_value < 0.001 else "**" if p_value < 0.01 else "*" if p_value < 0.05 else "ns"
            print(f"{comparison}: p = {p_value:.4f} {significance}")

        # Model efficiency analysis
        print("\nModel Efficiency Analysis:")
        print("-" * 50)
        neural_models = [model for model in results_df['Model'].unique() if 'Wavelet' not in model]
        for model in neural_models:
            model_data = results_df[results_df['Model'] == model]
            avg_snr = model_data['SNR'].mean()
            avg_params = model_data['Parameter_Count'].mean() / 1e6
            avg_size = model_data['Model_Size_MB'].mean()
            efficiency = avg_snr / avg_params if avg_params > 0 else 0
            print(f"{model}: Avg SNR={avg_snr:.2f}dB, Avg Params={avg_params:.2f}M, "
                  f"Avg Size={avg_size:.2f}MB, Efficiency={efficiency:.2f} dB/Mparam")

    print("\n" + "="*80)
    print("Model comparison training completed!")
    print(f"✓ All results and figures saved in: {os.path.abspath(results_dir)}")
    print("✓ Frequency spectrum plots saved for ALL models")
    print("✓ Best model weights (.pth files) saved for ALL neural network models")
    print("✓ All dimensionality issues fixed")
    print("="*80)

if __name__ == "__main__":
    main()